In [1]:
# Cell 1 — Setup
import sys, os

PROJECT_PATH = "/kaggle/input/datasets/evanlukedsouza/fedcrohn"
sys.path.append(PROJECT_PATH)
os.chdir(PROJECT_PATH)

print("CWD:", os.getcwd())
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
import numpy as np
import scipy
print("numpy:", np.__version__)
print("scipy:", scipy.__version__)

CWD: /kaggle/input/datasets/evanlukedsouza/fedcrohn
GPU available: True
GPU: Tesla T4
numpy: 2.0.2
scipy: 1.16.3


In [2]:
# Cell 2 — Verify files & imports
import os, pickle

def load_pkl(path):
    with open(path, 'rb') as f:
        content = f.read()
    return pickle.loads(content.replace(b'\r\n', b'\n'), encoding='latin1')

for f in [
    "marshalledP3/totGeneSet.m.min0",
    "marshalledP3/cagi2.multianno.missenseFalse.RegionsNone.m.min0",
    "marshalledP3/cagi3.multianno.missenseFalse.RegionsNone.m.min0",
    "marshalledP3/cagi4.multianno.missenseFalse.RegionsNone.m.min0",
    "marshalledP3/adj_cache.pkl",
    "phenopediaCrohnGenes/CrohnGenes.txt",
    "string_db/9606.protein.info.v12.0.txt",
    "string_db/9606.protein.links.v12.0.txt",
]:
    exists = os.path.exists(f)
    size = f"{os.path.getsize(f)/1e6:.1f} MB" if exists else "MISSING"
    print(f"{'OK' if exists else 'MISSING':8s} {size:10s}  {f}")

print("\nTesting imports...")
from sources.GATmodel import GATCrohnModel
from sources.buildGeneGraph import build_adj_from_string, build_adj_phenopedia
from sources.FedExplainer import LocalExplainer, GlobalExplainer
from sources.PersonalisedFL import PersonalizedGATCrohn
import sources.GraphConv as GCN
print("All imports OK")

gs = load_pkl("marshalledP3/totGeneSet.m.min0")
print(f"Gene set size: {len(gs)}")

OK       0.0 MB      marshalledP3/totGeneSet.m.min0
OK       11.8 MB     marshalledP3/cagi2.multianno.missenseFalse.RegionsNone.m.min0
OK       24.3 MB     marshalledP3/cagi3.multianno.missenseFalse.RegionsNone.m.min0
OK       27.4 MB     marshalledP3/cagi4.multianno.missenseFalse.RegionsNone.m.min0
OK       1.9 MB      marshalledP3/adj_cache.pkl
OK       0.0 MB      phenopediaCrohnGenes/CrohnGenes.txt
OK       6.3 MB      string_db/9606.protein.info.v12.0.txt
OK       630.9 MB    string_db/9606.protein.links.v12.0.txt

Testing imports...
All imports OK
Gene set size: 691


In [3]:
# Cell 3 — Load gene graph from cache
from sources.buildGeneGraph import build_adj_from_string, build_adj_phenopedia
from sources.readPhenopedia import readPhenopedia
import numpy as np

geneList = sorted(load_pkl("marshalledP3/totGeneSet.m.min0"))
weightPhenoPGenes, _ = readPhenopedia("phenopediaCrohnGenes/CrohnGenes.txt")

adj = build_adj_from_string(
    geneList,
    string_links_path="string_db/9606.protein.links.v12.0.txt",
    string_info_path="string_db/9606.protein.info.v12.0.txt",
    threshold=700,
    cache_path="marshalledP3/adj_cache.pkl"
)

if adj is None or np.array_equal(adj, np.eye(len(geneList))):
    print("Falling back to Phenopedia-based adjacency")
    adj = build_adj_phenopedia(geneList, weightPhenoPGenes)

print(f"Adjacency matrix shape: {adj.shape}")
print(f"Non-zero edges: {int((adj > 0).sum() - adj.shape[0])}")
print(f"Density: {(adj > 0).mean():.4f}")

Found 691 associated genes.
Loading adjacency matrix from cache: marshalledP3/adj_cache.pkl
Adjacency matrix shape: (691, 691)
Non-zero edges: 6390
Density: 0.0148


In [4]:
# Cell 4 — Imports & globals
import sys, os, pickle, random
import numpy as np
import torch as t
from collections import OrderedDict
from typing import List, Tuple
from sklearn.model_selection import StratifiedKFold

from sources.iongreen2_analysisPaper import scanGenes, buildFeatVect, buildVectorGeneWise, checkVectors
from sources.readPhenopedia import readPhenopedia
from sources import utils as U
from sources.GATmodel import GATCrohnModel
from sources.buildGeneGraph import build_adj_from_string, build_adj_phenopedia
from sources.FedExplainer import LocalExplainer, GlobalExplainer
from sources.PersonalisedFL import PersonalizedGATCrohn
import sources.Constants as CONST

DEVICE = t.device("cuda" if t.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

MODEL_DIR = "/kaggle/working/models"
os.makedirs(MODEL_DIR, exist_ok=True)

MIN_REFERENCE_NUM = 0
DBs = [
    "marshalledP3/cagi2.multianno.missenseFalse.RegionsNone.m.min0",
    "marshalledP3/cagi3.multianno.missenseFalse.RegionsNone.m.min0",
    "marshalledP3/cagi4.multianno.missenseFalse.RegionsNone.m.min0",
]

global_explainer = None
# Reproducibility
import random as _random
_random.seed(42)
np.random.seed(42)
t.manual_seed(42)
t.cuda.manual_seed_all(42)
print("Random seeds set to 42")


Using device: cuda


In [5]:
# Cell 5 — Patch GraphConv (read-only path, verbose removed, GPU enabled)
import sources.GraphConv as _gcn
import torch as _t
import os
from sys import stdout

def _patched_fit(self, originalX, originalY, epochs=50, batch_size=20,
                 save_model_every=10, warmStart=0, weight_decay=0.001,
                 learning_rate=1e-3, silent=False):
    import numpy as np
    X_arr = np.array(originalX, dtype=np.float32)
    Y_arr = np.array(originalY, dtype=np.int64)
    X_tensor = _t.FloatTensor(X_arr).to(DEVICE)
    Y_tensor = _t.LongTensor(Y_arr).to(DEVICE)
    self.model.to(DEVICE)
    self.model.train()
    if not silent:
        print("Start training")
    # Class-weighted loss — fixes sensitivity collapse on 67/33 imbalance
    n_pos = int(sum(originalY))
    n_neg = len(originalY) - n_pos
    w_pos = len(originalY) / (2.0 * n_pos) if n_pos > 0 else 1.0
    w_neg = len(originalY) / (2.0 * n_neg) if n_neg > 0 else 1.0
    class_weights = _t.FloatTensor([w_neg, w_pos]).to(DEVICE)
    lossfn = _gcn.LossWrapperCE(
        _t.nn.CrossEntropyLoss(weight=class_weights, ignore_index=-1, reduction='sum'),
        dummyColumn=True
    )
    self.learning_rate = learning_rate
    optimizer = _t.optim.RMSprop(
        self.model.parameters(), lr=self.learning_rate, weight_decay=weight_decay
    )
    scheduler = _t.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.3, patience=10,
        threshold=0.0001, threshold_mode='rel', cooldown=0, min_lr=0, eps=1e-08
    )
    os.makedirs(MODEL_DIR, exist_ok=True)
    ofp = open(os.path.join(MODEL_DIR, self.model.name + ".TrainLog"), "w")
    n = X_tensor.size(0)
    e = 1 + warmStart
    while e < epochs + warmStart:
        errTot = 0.0
        indices = _t.randperm(n)
        for start in range(0, n, batch_size):
            idx = indices[start:start + batch_size]
            x_b = X_tensor[idx]
            y_b = Y_tensor[idx]
            optimizer.zero_grad()
            yp = self.model.forward(x_b)
            if len(yp.shape) == 1:
                continue
            loss = lossfn(yp, y_b)
            loss.backward()
            optimizer.step()
            errTot += loss.item()
        if not silent:
            print(f" epoch {e}, ERRORTOT: {errTot:.4f}")
        scheduler.step(errTot)
        stdout.flush()
        e += 1
    _t.save(self.model, os.path.join(MODEL_DIR, self.model.name + ".final.t"))
    ofp.close()

def _patched_predict(self, X, Y=None, batch_size=-1, GET_ACT=False):
    import numpy as np
    self.model.to(DEVICE)
    self.model.eval()
    X_arr = np.array(X, dtype=np.float32)
    x_tensor = _t.FloatTensor(X_arr).to(DEVICE)
    with _t.no_grad():
        if GET_ACT:
            gene_act, yp = self.model.forward(x_tensor, GET_ACT=True)
            yp = _t.nn.functional.sigmoid(yp)
            return gene_act.cpu().squeeze().tolist(), yp.cpu().squeeze().tolist()
        else:
            yp = _t.nn.functional.sigmoid(self.model.forward(x_tensor))
            result = yp.cpu().squeeze()
            if result.dim() == 0:
                return [result.item()]
            return result.tolist()

_gcn.NNwrapper.fit     = _patched_fit
_gcn.NNwrapper.predict = _patched_predict
print("GraphConv patched — read-only fix, verbose removed, GPU enabled")

GraphConv patched — read-only fix, verbose removed, GPU enabled


In [6]:
# Cell 6 — Patch GATLayer: memory-efficient per-head attention
#           Fixes get_gene_importance to match [B,N,N] alpha shape (not [B,N,N,H])
import torch
import torch.nn.functional as F
from sources.GATmodel import GATLayer, GATCrohnModel

def _efficient_gat_forward(self, x, adj):
    """
    Per-head loop: peak alloc [B,N,N] per head instead of [B,N,N,H] all at once.
    B=4, N=691: ~7 MB per head vs ~3.6 GB original.
    """
    B, N, _ = x.shape
    h = self.W(x).view(B, N, self.num_heads, self.out_features)

    head_outputs = []
    last_alpha = None

    for head_idx in range(self.num_heads):
        h_head = h[:, :, head_idx, :]                          # [B, N, F]
        a_head = self.a[head_idx]                              # [2F]
        h_i = h_head.unsqueeze(2).expand(B, N, N, self.out_features)
        h_j = h_head.unsqueeze(1).expand(B, N, N, self.out_features)
        e = self.leakyrelu(
            torch.cat([h_i, h_j], dim=-1).matmul(a_head)
        )                                                      # [B, N, N]
        mask  = (adj == 0).unsqueeze(0)                       # [1, N, N]
        e     = e.masked_fill(mask, float('-inf'))
        alpha = F.softmax(e, dim=2)                           # [B, N, N]
        alpha = self.dropout(alpha)
        out_head = torch.bmm(alpha, h_head)                   # [B, N, F]
        head_outputs.append(out_head)
        last_alpha = alpha.detach()
        del h_i, h_j, e, alpha, out_head
        torch.cuda.empty_cache()

    out = torch.stack(head_outputs, dim=0).mean(dim=0)        # [B, N, F]
    return out, last_alpha                                     # last_alpha: [B,N,N]


def _fixed_get_gene_importance(self):
    """
    attention_weights shape after patch: [B, N, N]  (3-D, NOT 4-D).
    Mean over batch dim -> [N, N], then sum attention received per gene -> [N].
    """
    if self.attention_weights is None:
        return None
    return self.attention_weights.mean(dim=0).sum(dim=0)      # [N]


GATLayer.forward                  = _efficient_gat_forward
GATCrohnModel.get_gene_importance = _fixed_get_gene_importance
print("GATLayer patched    — memory-efficient per-head attention")
print("get_gene_importance — fixed for 3-D [B,N,N] alpha")

GATLayer patched    — memory-efficient per-head attention
get_gene_importance — fixed for 3-D [B,N,N] alpha


In [7]:
# Cell 6b — Fix GATCrohnModel.forward() signature (adj is a buffer, not an argument)
import torch.nn.functional as F
from sources.GATmodel import GATCrohnModel

def _fixed_forward(self, x, GET_ACT=False):
    """
    adj is registered as a buffer inside the model — no need to pass it externally.
    This matches the call signature expected by NNwrapper: model.forward(x)
    """
    out, attn1 = self.gat1(x, self.adj)
    out = F.leaky_relu(out)
    out, attn2 = self.gat2(out, self.adj)

    # Store attention weights for get_gene_importance()
    self.attention_weights = attn2.detach()

    out_flat = out.view(out.size(0), -1)   # [B, N*hidden]
    pred = self.classifier(out_flat)        # [B, 1]

    if GET_ACT:
        return out, pred
    return pred

GATCrohnModel.forward = _fixed_forward
print("GATCrohnModel.forward patched — adj from buffer, attention_weights stored")

GATCrohnModel.forward patched — adj from buffer, attention_weights stored


In [8]:
# Cell 7 — Helper functions
def load_pkl(path):
    with open(path, 'rb') as f:
        content = f.read()
    return pickle.loads(content.replace(b'\r\n', b'\n'), encoding='latin1')

def _set_params(model, params):
    params_dict = zip(model.state_dict().keys(), params)
    state_dict = OrderedDict({k: t.from_numpy(np.copy(v)) for k, v in params_dict})
    model.load_state_dict(state_dict, strict=True)

def splitData(x, y, folds, shuffle=True):
    X, Y = np.array(x), np.array(y)
    skf = StratifiedKFold(n_splits=folds, shuffle=shuffle)
    datasets = []
    for _, test in skf.split(X, Y):
        datasets.append((list(X[test]), list(Y[test])))
    assert sum(len(d[0]) for d in datasets) == len(x)
    return datasets

In [9]:
# Cell 8 — main()
def main(CV_FOLDS=5, use_personalized=False, num_rounds=5, epochs_per_client=20):
    global global_explainer

    NUM_CLIENTS = CV_FOLDS - 1

    weightPhenoPGenes, phenoPGenes = readPhenopedia("phenopediaCrohnGenes/CrohnGenes.txt")
    geneList = sorted(load_pkl(f"marshalledP3/totGeneSet.m.min{MIN_REFERENCE_NUM}"))

    adj = build_adj_from_string(
        geneList,
        string_links_path="string_db/9606.protein.links.v12.0.txt",
        string_info_path="string_db/9606.protein.info.v12.0.txt",
        threshold=700,
        cache_path="marshalledP3/adj_cache.pkl"
    )
    if adj is None or np.array_equal(adj, np.eye(len(geneList))):
        print("Falling back to Phenopedia-based adjacency")
        adj = build_adj_phenopedia(geneList, weightPhenoPGenes)

    DATAX, DATAY = [], []
    HX = {}
    for D in DBs:
        db = load_pkl(D)
        for sampleName, (exome, label) in db.items():
            geneDB = scanGenes(exome.items(), geneList)
            HX[sampleName] = (buildVectorGeneWise(geneDB, geneList, weightPhenoPGenes, None), label)
        X, Y = buildFeatVect(HX, list(HX.keys()))
        DATAX += X
        DATAY += Y

    datasets = splitData(DATAX, DATAY, CV_FOLDS)
    global_explainer = GlobalExplainer(geneList)
    totRes = {"sen": [], "spe": [], "pre": [], "mcc": [], "auc": [], "auprc": []}

    for fold_i in range(CV_FOLDS):
        print(f"\n{'='*50}")
        print(f"FOLD {fold_i+1}/{CV_FOLDS}")

        testX, testY = datasets[fold_i]
        trainsets    = [datasets[j] for j in range(CV_FOLDS) if j != fold_i]

        genesize   = len(testX[0][0])
        numGenes   = len(testX[0])
        global_net = GATCrohnModel(genesize, numGenes, adj, geneList)
        global_net.to(DEVICE)
        global_params = [v.cpu().numpy() for _, v in global_net.state_dict().items()]

        for rnd in range(1, num_rounds + 1):
            print(f"  Round {rnd}/{num_rounds}", end="  ", flush=True)

            new_params_list = []
            client_sizes    = []

            for cid, (cX, cY) in enumerate(trainsets):
                from sources.GraphConv import NNwrapper
                client_net = GATCrohnModel(genesize, numGenes, adj, geneList)
                client_net.to(DEVICE)
                _set_params(client_net, global_params)
                wrapper = NNwrapper(client_net)
                wrapper.fit(cX, cY,
                            epochs=epochs_per_client,
                            batch_size=4,
                            weight_decay=1,
                            learning_rate=1e-3,
                            silent=True)
                new_params_list.append(
                    [v.cpu().numpy() for _, v in client_net.state_dict().items()]
                )
                client_sizes.append(len(cX))
                del client_net, wrapper
                t.cuda.empty_cache()

            # FedAvg
            total = sum(client_sizes)
            global_params = [
                sum(new_params_list[c][i] * (client_sizes[c] / total)
                    for c in range(NUM_CLIENTS))
                for i in range(len(global_params))
            ]

            # Server evaluation — small batch to stay within GPU memory
            _set_params(global_net, global_params)
            from sources.GraphConv import NNwrapper
            wrapper = NNwrapper(global_net)
            Yp = wrapper.predict(testX, batch_size=8)
            sen, spe, acc, bac, pre, mcc, auc, auprc = U.getScoresSVR(
                Yp, testY, threshold=None, invert=True,
                PRINT=False, CURVES=False, SAVEFIG=None
            )
            print(f"MCC={mcc:.3f}  AUC={auc:.3f}", flush=True)

            imp = global_net.get_gene_importance()
            if imp is not None:
                global_explainer.round_importances.append(imp.cpu().numpy())

            del wrapper
            t.cuda.empty_cache()

        totRes["sen"].append(sen)
        totRes["spe"].append(spe)
        totRes["pre"].append(pre)
        totRes["mcc"].append(mcc)
        totRes["auc"].append(auc)
        totRes["auprc"].append(auprc)
        print(f"Fold {fold_i+1} done — MCC={mcc:.3f}  AUC={auc:.3f}")

        del global_net
        t.cuda.empty_cache()

    print("\n=== FINAL RESULTS ===")
    for metric, vals in totRes.items():
        print(f"{metric}: {np.mean(vals):.4f} \u00b1 {np.std(vals):.4f}")

    os.makedirs("/kaggle/working/results", exist_ok=True)
    global_explainer.save_report("/kaggle/working/results/gene_importance.txt")

    # Save fold results to CSV
    import pandas as pd
    results_df = pd.DataFrame(totRes)
    results_df.to_csv("/kaggle/working/results/fold_results.csv", index=False)

    # Gene importance analysis vs known Crohn's GWAS genes
    known_crohn = {
        'NOD2','IL23R','ATG16L1','CARD9','JAK2','STAT3','IL10',
        'TNFSF15','MST1','PTPN22','IRGM','NKX2-3','LRRK2','CDKAL1',
        'PTGER4','SLC22A5','ICOSLG','TNFRSF6B','RNF186','SMAD3'
    }
    try:
        imp_df = pd.read_csv("/kaggle/working/results/gene_importance.txt", sep="\t")
        imp_df['known_crohn'] = imp_df['Gene'].isin(known_crohn)
        top20 = imp_df.head(20)
        overlap = top20[top20['known_crohn']]
        print(f"\nKnown Crohn's genes in top 20: {len(overlap)}/20")
        if len(overlap):
            print(overlap[['Rank','Gene','Importance']].to_string(index=False))
    except Exception as ex:
        print(f"Gene analysis skipped: {ex}")

    print("\nSummary statistics:")
    print(results_df.describe().round(3))
    print("Results saved to /kaggle/working/results/")

In [10]:
# Cell 9 — Run
main(CV_FOLDS=5, use_personalized=False, num_rounds=5, epochs_per_client=50)

Found 691 associated genes.
Loading adjacency matrix from cache: marshalledP3/adj_cache.pkl

FOLD 1/5
  Round 1/5  MCC=-0.318  AUC=0.597
  Round 2/5  MCC=-0.318  AUC=0.602
  Round 3/5  MCC=-0.318  AUC=0.619
  Round 4/5  MCC=-0.318  AUC=0.610
  Round 5/5  MCC=-0.318  AUC=0.613
Fold 1 done — MCC=-0.318  AUC=0.613

FOLD 2/5
  Round 1/5  MCC=-0.219  AUC=0.595
  Round 2/5  MCC=-0.172  AUC=0.597
  Round 3/5  MCC=-0.172  AUC=0.591
  Round 4/5  MCC=-0.157  AUC=0.596
  Round 5/5  MCC=-0.172  AUC=0.596
Fold 2 done — MCC=-0.172  AUC=0.596

FOLD 3/5
  Round 1/5  MCC=-0.423  AUC=0.756
  Round 2/5  MCC=-0.351  AUC=0.736
  Round 3/5  MCC=-0.423  AUC=0.752
  Round 4/5  MCC=-0.405  AUC=0.751
  Round 5/5  MCC=-0.441  AUC=0.758
Fold 3 done — MCC=-0.441  AUC=0.758

FOLD 4/5
  Round 1/5  MCC=-0.318  AUC=0.640
  Round 2/5  MCC=-0.301  AUC=0.586
  Round 3/5  MCC=-0.301  AUC=0.596
  Round 4/5  MCC=-0.301  AUC=0.599
  Round 5/5  MCC=-0.279  AUC=0.634
Fold 4 done — MCC=-0.279  AUC=0.634

FOLD 5/5
  Round 1/5  M